In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("results/ik_collision/20261109_224232/all.csv")
ok = df.pivot_table(index=["obstacle", "scenario_id"], columns="method",
                    values="feasible", aggfunc="first").all(axis=1)
paired = df.set_index(["obstacle", "scenario_id"]).loc[ok[ok].index].reset_index()

t = paired.pivot_table(index=["obstacle", "scenario_id"], columns="method", values="solve_time_s")
speedup = t.rdiv(t["fcl"], axis=0)
print(speedup.groupby(level="obstacle").median())

In [ ]:
ok = df.pivot_table(index=["obstacle", "scenario_id"], columns="method",
                    values="feasible", aggfunc="first").all(axis=1)
paired = df.set_index(["obstacle", "scenario_id"]).loc[ok[ok].index].reset_index()
paired["solve_time_ms"] = 1e3 * paired["solve_time_s"]

cols = ["solve_time_ms", "cost", "max_violation", "min_distance", "gap_at_q0"]
table = paired.groupby(["obstacle", "method"])[cols].median()
table.insert(0, "n", paired.groupby(["obstacle", "method"]).size())
print(table.to_string(float_format=lambda x: f"{x:.4g}"))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


ok = df.pivot_table(index=["obstacle", "scenario_id"], columns="method",
                    values="feasible", aggfunc="first").all(axis=1)
df = df.set_index(["obstacle", "scenario_id"]).loc[ok[ok].index].reset_index()

methods = ["fcl", "coal_default", "coal_polyak", "coal_nesterov"]
labels = ["FCL", "Coal", "Polyak", "Nesterov"]
colors = ["lightseagreen", "royalblue", "mediumpurple", "darkorange"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, obstacle in zip(axes, ["cube", "sphere_hull"]):
    d = df[df.obstacle == obstacle]
    bp = ax.boxplot([1e3 * d[d.method == m].solve_time_s for m in methods],
                    patch_artist=True, showfliers=True, medianprops=dict(color="black"))
    for box, c in zip(bp["boxes"], colors):
        box.set_facecolor(c)
    ax.set_ylim(top=1e3 * d.solve_time_s.quantile(0.99))
    ax.set_xticks(range(1, 5), labels)
    ax.set_title(obstacle)
axes[0].set_ylabel("IK solve time (ms)")


In [ ]:
fcl_time = table["solve_time_ms"].xs("fcl", level="method")
table["diff_vs_fcl_ms"] = table["solve_time_ms"] - fcl_time.reindex(table.index, level="obstacle")
print(table.to_string(float_format=lambda x: f"{x:.4g}"))

In [ ]:
fcl = paired.loc[paired.method == "fcl", ["obstacle", "scenario_id", "solve_time_s"]]
paired = paired.merge(fcl, on=["obstacle", "scenario_id"], suffixes=("", "_fcl"))
paired["diff_vs_fcl_ms"] = 1e3 * (paired["solve_time_s"] - paired["solve_time_s_fcl"])



In [ ]:

print(df[df["scenario_id"]==8])

In [ ]:
df[(df["solve_time_s"]>0.7) & (df["method"]=="fcl")]["scenario_id"]